In [ ]:

# plotting dymotree analysed anndata object
import scanpy as sc
import pandas as pd
import numpy as np
import anndata
import matplotlib.colors as mcolors
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt
tree = sc.read('./Fig2/1.Larry.DMT.bench.result/Larry_dmt.h5ad')
stem = sc.read('./Fig2/1.Larry.DMT.bench.result/HSPC_dmt.h5ad')

cytotrace = pd.read_csv('./Fig2/1.Larry.DMT.bench.result/CytoTrace.Score.csv')
tree.obs['CytoTrace_score'] = 1-cytotrace['x'].values
stem.obs['CytoTrace_score'] = tree.obs.loc[tree.obs['celltype']=='HSPC','CytoTrace_score'].values

gene = 'CytoTrace_score'
with rc_context({'figure.figsize': (6, 3)}):
    ax = sc.pl.violin(
        tree,
        gene,
        groupby='new_State',
        stripplot=False,
        inner='box',
        rotation=45,
        show=False
    )
    #plt.savefig(f"./Fig2/1.Larry.DMT.bench.result/Larry.fate_state.CytoTrace.png",bbox_inches='tight')
    plt.show()

In [ ]:
import pandas as pd
from scipy.stats import entropy
import numpy as np
prob_values = stem.obs[['Monocyte_fate', 'Neutrophil_fate']].values
stem.obs['Entropy'] = entropy(prob_values, axis=1, base=2)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np

df = stem.obs[['Entropy','CytoTrace_score']]
# ----------------------------------------------------


# --- 2. 计算相关性 (r) 和 P 值 ---
# 使用 scipy.stats.pearsonr
# 它返回 (相关系数r, P值)
r, p = stats.pearsonr(df['Entropy'], df['CytoTrace_score'])

print(f"Pearson's r (相关系数): {r:.4f}")
print(f"P-value (P值): {p:.4f}")


# --- 3. 绘制散点图 ---
# 设置画布大小
plt.figure(figsize=(6, 5))

# 使用 seaborn 的 regplot，它会自动绘制散点图和回归线
ax = sns.regplot(
    data=df,
    x='Entropy',
    y='CytoTrace_score',
    scatter_kws={'s': 20, 'alpha': 0.8}, # 设置点的大小和透明度
    line_kws={'color': 'red', 'linestyle': '--'} # 设置回归线的颜色和样式
)

# --- 4. 在图上添加注释 (R值和P值) ---

# 准备注释文本
# (我们对P值进行格式化，如果太小，就显示 < 0.001)
p_text = f"P < 0.001" if p < 0.001 else f"P-value = {p:.3f}"
annotation_text = f"Pearson's r = {r:.2f}\n{p_text}"

# 使用 plt.text() 添加文本
# transform=ax.transAxes 表示使用相对于“坐标轴”的坐标系 (0,0)是左下角, (1,1)是右上角
# 这样无论你的数据范围如何，文本总是在固定的位置
plt.text(
    0.05, 0.95, # 放在左上角 (x=5%, y=95%)
    annotation_text,
    transform=ax.transAxes,
    fontsize=12,
    verticalalignment='top', # 垂直对齐方式
    # 添加一个白色背景框，使文本更清晰
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
)


# --- 5. 美化和显示图像 ---
plt.xlabel('CytoTrace_Score', fontsize=12)
plt.ylabel('Entropy', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6) # 添加网格线
plt.tight_layout() # 自动调整布局
plt.savefig(f"./Fig2/1.Larry.DMT.bench.result/Larry.Entropy.CytoTrace.corr.pdf",bbox_inches='tight')
plt.show()
